[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/ext-02-backprop-by-hand-and-vanishing-gradients.ipynb)

# Advanced Discussion 2 — backprop by hand, four equations, forward vs reverse mode, vanishing gradients

Reproduces the chapter 2 advanced discussion: a worked 2-2-1 backward pass checked against autograd, matrix-form backprop for any depth, dual numbers, and a measured vanishing-gradient table.

**Optional advanced-discussion lab** for Hands-On Deep Learning with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier (CPU is enough) with built-in datasets, so no download can fail.

Offline fallback: everything is generated locally.

In [ ]:
import numpy as np, torch, torch.nn as nn

## 1. hand-worked example: 2-2-1 network, one example, sigmoid units, squared-error loss

In [ ]:
x = np.array([[0.5, -1.0]]); y = np.array([[1.0]])
W1 = np.array([[0.1, -0.2], [0.4, 0.3]]); b1 = np.array([[0.0, 0.1]])
W2 = np.array([[0.7], [-0.5]]); b2 = np.array([[0.05]])
sig = lambda z: 1 / (1 + np.exp(-z))
z1 = x @ W1 + b1; a1 = sig(z1); z2 = a1 @ W2 + b2; a2 = sig(z2); L = 0.5 * ((a2 - y) ** 2).sum()
print("z1", z1.round(4).tolist(), "a1", a1.round(4).tolist(), "z2", z2.round(4).tolist(), "a2 %.4f" % a2.item(), "loss %.4f" % L)
d2 = (a2 - y) * a2 * (1 - a2)                       # delta at the output
dW2 = a1.T @ d2; db2 = d2.sum(0, keepdims=True)
d1 = (d2 @ W2.T) * a1 * (1 - a1)                    # delta pushed back through W2, times sigma'
dW1 = x.T @ d1; db1 = d1.sum(0, keepdims=True)
print("delta2", d2.round(5).tolist(), "delta1", d1.round(5).tolist())
print("dW2", dW2.round(5).tolist(), "dW1", dW1.round(5).tolist())
# check against autograd
tW1, tb1, tW2, tb2 = [torch.tensor(a, requires_grad=True) for a in (W1, b1, W2, b2)]
tx, ty = torch.tensor(x), torch.tensor(y)
out = torch.sigmoid(torch.sigmoid(tx @ tW1 + tb1) @ tW2 + tb2); (0.5 * ((out - ty) ** 2).sum()).backward()
print("max abs difference vs autograd:", max(np.abs(dW1 - tW1.grad.numpy()).max(), np.abs(dW2 - tW2.grad.numpy()).max(), np.abs(db1 - tb1.grad.numpy()).max(), np.abs(db2 - tb2.grad.numpy()).max()))

## 2. general matrix-form backprop for an L-layer MLP, checked against autograd

In [ ]:
def forward(params, X):
    acts, zs = [X], []
    for i, (W, b) in enumerate(params):
        z = acts[-1] @ W + b; zs.append(z)
        acts.append(np.tanh(z) if i < len(params) - 1 else z)          # tanh hidden, linear output
    return acts, zs
def backward(params, acts, zs, Y):
    n = len(Y); grads = [None] * len(params)
    delta = (acts[-1] - Y) / n                                            # dL/dz for L = 1/(2n) * sum (out - y)^2, linear output
    for l in reversed(range(len(params))):
        W, b = params[l]
        grads[l] = (acts[l].T @ delta, delta.sum(0, keepdims=True))
        if l > 0:
            delta = (delta @ W.T) * (1 - np.tanh(zs[l - 1]) ** 2)         # delta^{l-1} = (delta^l W^T) * tanh'(z^{l-1})
    return grads
rng = np.random.RandomState(0)
dims = [4, 6, 5, 2]
params = [(rng.randn(a, b) * 0.5, rng.randn(1, b) * 0.1) for a, b in zip(dims[:-1], dims[1:])]
X, Y = rng.randn(7, 4), rng.randn(7, 2)
acts, zs = forward(params, X); g = backward(params, acts, zs, Y)
tp = [(torch.tensor(W, requires_grad=True), torch.tensor(b, requires_grad=True)) for W, b in params]
h = torch.tensor(X)
for i, (W, b) in enumerate(tp): h = h @ W + b; h = torch.tanh(h) if i < len(tp) - 1 else h
(0.5 * ((h - torch.tensor(Y)) ** 2).sum() / len(Y)).backward()
print("matrix-form backprop vs autograd, max abs diff:", max(max(np.abs(gw - W.grad.numpy()).max(), np.abs(gb - b.grad.numpy()).max()) for (gw, gb), (W, b) in zip(g, tp)))

## 3. forward mode (dual numbers) vs reverse mode: passes needed

In [ ]:
class Dual:
    def __init__(s, v, d): s.v, s.d = v, d
    def __add__(s, o): o = o if isinstance(o, Dual) else Dual(o, 0.0); return Dual(s.v + o.v, s.d + o.d)
    def __mul__(s, o): o = o if isinstance(o, Dual) else Dual(o, 0.0); return Dual(s.v * o.v, s.v * o.d + s.d * o.v)
    __radd__, __rmul__ = __add__, __mul__
def f(x, y, z): return x * y + y * z + x * z * x                       # one scalar output, three inputs
pt = (2.0, 3.0, 5.0)
fwd = [f(*[Dual(p, 1.0 if i == j else 0.0) for j, p in enumerate(pt)]).d for i in range(3)]      # one forward pass PER input
tp_ = [torch.tensor(p, requires_grad=True) for p in pt]; f(*tp_).backward()                     # one reverse pass for all inputs
print("forward mode (3 passes):", fwd, "| reverse mode (1 pass):", [t.grad.item() for t in tp_])

## 4. vanishing gradients: gradient norm reaching the FIRST layer of a deep MLP

In [ ]:
def first_layer_grad(act, depth, init, seed=0):
    torch.manual_seed(seed)
    layers = []
    for _ in range(depth):
        l = nn.Linear(64, 64)
        if init == "he": nn.init.kaiming_normal_(l.weight, nonlinearity="relu")
        elif init == "xavier": nn.init.xavier_normal_(l.weight)
        else: nn.init.normal_(l.weight, std=0.05)
        nn.init.zeros_(l.bias); layers += [l, act()]
    net = nn.Sequential(*layers, nn.Linear(64, 1))
    x = torch.randn(256, 64); net(x).pow(2).mean().backward()
    return layers[0].weight.grad.norm().item(), layers[-2].weight.grad.norm().item()
print("depth 10 | grad norm at layer 1 vs layer 10")
for name, act, init in (("sigmoid, small random init", nn.Sigmoid, "small"), ("tanh, Xavier", nn.Tanh, "xavier"), ("ReLU, small random init", nn.ReLU, "small"), ("ReLU, He init", nn.ReLU, "he")):
    a, b = first_layer_grad(act, 10, init); print("  %-28s first %.2e   last %.2e   ratio %.2e" % (name, a, b, a / b))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.